# Validación analítica de datos para modelado de churn

**Caso:** BookStore Online  
**Curso:** SI7002 – Sistemas de Aprendizaje Automático  
**Enfoque:** DataOps + MLOps  
**Propósito del notebook:** validar los datos generados por el pipeline antes de avanzar hacia el entrenamiento de modelos.

Este notebook se ejecuta después de haber corrido la Step Function que materializa las capas:

```text
raw → curated → features
```

Este dataset es el resultado de la ejecución completa de la State Machine definida en la guía de laboratorio. Por tanto, antes de iniciar cualquier proceso de entrenamiento, se requiere validar que los datos generados sean consistentes, completos y adecuados para un escenario de modelado de churn.

## 1. Nota metodológica importante

Este notebook **NO forma parte del pipeline de transformación**.

Todas las transformaciones principales deben ejecutarse mediante jobs de AWS Glue, orquestados desde AWS Step Functions. En consecuencia, el notebook no debe ser usado para corregir manualmente datos, rehacer transformaciones o construir un pipeline paralelo.

El propósito de este notebook es exclusivamente:

1. Validar la salida generada por el pipeline.
2. Analizar la calidad de los datos en la capa `features`.
3. Comprender el significado de las variables disponibles.
4. Evaluar si el dataset puede ser usado de manera responsable en una etapa posterior de Machine Learning.

Esta distinción es fundamental en un enfoque DataOps/MLOps: el pipeline produce datos reproducibles; el notebook permite inspeccionarlos, analizarlos y tomar decisiones informadas antes del modelado.

## 2. Configuración de la sesión de AWS Glue

La siguiente celda está pensada para ejecutarse en un notebook de AWS Glue Interactive Sessions.

> Si ya tiene una sesión activa, puede omitir o ajustar esta celda según la configuración de su entorno.

In [9]:
%stop_session
%%configure -f
{
  "--job-language": "python",
  "--enable-glue-datacatalog": "true",
  "--conf": "spark.sql.sources.partitionOverwriteMode=dynamic",
  "--TempDir": "s3://data1-bkstore/tmp/glue-notebooks/"
}

Stopping session: 3512dc13-9148-44df-b154-9f7a1f93a3f0
Stopped session.
The following configurations have been updated: {'--job-language': 'python', '--enable-glue-datacatalog': 'true', '--conf': 'spark.sql.sources.partitionOverwriteMode=dynamic', '--TempDir': 's3://data1-bkstore/tmp/glue-notebooks/'}


## 3. Crear contexto de Glue y Spark

Se inicializa el contexto de Glue y la sesión Spark.  
Esta sesión será usada para leer los datos generados por los jobs de transformación.

### Inicialización del contexto de ejecución

A diferencia de un entorno local de PySpark, donde el `SparkSession` debe ser creado manualmente, en este laboratorio se utiliza AWS Glue como entorno de ejecución.

En este contexto, la sesión de Spark es gestionada por Glue, y se accede a ella a través del `GlueContext`. Esto permite integrar el procesamiento de datos con los servicios administrados de AWS y mantener coherencia con el pipeline definido en las secciones anteriores.

Por esta razón, en este notebook no se crea un `SparkSession` desde cero, sino que se obtiene a partir del contexto de Glue.

In [1]:
import sys
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql import types as T

from awsglue.context import GlueContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

spark

Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: c9884785-244b-42ee-bd6c-461c2dcc25d4
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
--job-language python
--conf spark.sql.sources.partitionOverwriteMode=dynamic
--TempDir s3://data1-bkstore/tmp/glue-notebooks/
Waiting for session c9884785-244b-42ee-bd6c-461c2dcc25d4 to get into ready status...
Session c9884785-244b-42ee-bd6c-461c2dcc25d4 has been created.


## 4. Parámetros del laboratorio

Ajuste el nombre del bucket según la configuración usada en su cuenta de AWS Academy.

La ruta principal que se analizará en este notebook corresponde a la capa `features`, generada a partir de los datos en `curated`.

In [2]:
# ============================================================
# Parámetros del laboratorio
# ============================================================

BUCKET_NAME = "data1-bkstore"

CURATED_BASE_PATH = f"s3://{BUCKET_NAME}/curated/"
FEATURES_PATH = f"s3://{BUCKET_NAME}/features/customer_features/"

LABEL_COL = "churn_label_60d"

print("Curated path:", CURATED_BASE_PATH)
print("Features path:", FEATURES_PATH)

Curated path: s3://data1-bkstore/curated/
Features path: s3://data1-bkstore/features/customer_features/


## 5. Lectura de la capa features

En esta sección se carga el dataset `customer_features`, el cual contiene una fila por cliente y variables derivadas para análisis posterior de churn.

Esta lectura permite validar si la Step Function y los jobs de Glue generaron correctamente la salida esperada.

In [3]:
features_df = spark.read.parquet(FEATURES_PATH)

features_df.printSchema()
features_df.show(10, truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- first_order_date: date (nullable = true)
 |-- last_order_date: date (nullable = true)
 |-- days_since_last_order: integer (nullable = true)
 |-- days_as_customer: integer (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_spent: decimal(22,2) (nullable = true)
 |-- avg_order_value: decimal(16,6) (nullable = true)
 |-- distinct_books_purchased: long (nullable = true)
 |-- total_items_purchased: long (nullable = true)
 |-- preferred_category: string (nullable = true)
 |-- churn_label_60d: integer (nullable = true)
 |-- feature_timestamp: timestamp (nullable = true)
 |-- snapshot_date: date (nullable = true)

+------------------------------------+------------+--------+----------------+---------------+---------------------+----------------+------------+-----------+---------------+------------------------+---------------------+------------------

## 6. Revisión básica del dataset

Antes de avanzar hacia cualquier proceso de modelado, es necesario responder preguntas mínimas sobre el dataset:

- ¿Cuántos registros existen?
- ¿Cuántas columnas tiene?
- ¿Existe la variable objetivo?
- ¿La distribución de la etiqueta parece razonable?

In [4]:
total_rows = features_df.count()
total_cols = len(features_df.columns)

print(f"Total de registros: {total_rows}")
print(f"Total de columnas: {total_cols}")

print("Columnas disponibles:")
for c in features_df.columns:
    print("-", c)

Total de registros: 403
Total de columnas: 16
Columnas disponibles:
- customer_id
- city
- country
- first_order_date
- last_order_date
- days_since_last_order
- days_as_customer
- total_orders
- total_spent
- avg_order_value
- distinct_books_purchased
- total_items_purchased
- preferred_category
- churn_label_60d
- feature_timestamp
- snapshot_date


In [5]:
if LABEL_COL in features_df.columns:
    features_df.groupBy(LABEL_COL).count().orderBy(LABEL_COL).show()
else:
    print(f"No se encontró la columna objetivo: {LABEL_COL}")

+---------------+-----+
|churn_label_60d|count|
+---------------+-----+
|              0|  183|
|              1|  220|
+---------------+-----+


## 7. Validación de valores nulos

La presencia de valores nulos puede indicar problemas en la transformación desde `curated` hacia `features`, errores de integración entre entidades o ausencia de información histórica para ciertos clientes.

En esta sección se calcula el número y porcentaje de valores nulos por columna.

In [6]:
null_counts = features_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in features_df.columns
])

null_counts.show(truncate=False)

+-----------+----+-------+----------------+---------------+---------------------+----------------+------------+-----------+---------------+------------------------+---------------------+------------------+---------------+-----------------+-------------+
|customer_id|city|country|first_order_date|last_order_date|days_since_last_order|days_as_customer|total_orders|total_spent|avg_order_value|distinct_books_purchased|total_items_purchased|preferred_category|churn_label_60d|feature_timestamp|snapshot_date|
+-----------+----+-------+----------------+---------------+---------------------+----------------+------------+-----------+---------------+------------------------+---------------------+------------------+---------------+-----------------+-------------+
|0          |0   |0      |0               |0              |0                    |0               |0           |0          |0              |0                       |0                    |0                 |0              |0                

In [7]:
total = features_df.count()

null_percent_exprs = [
    (F.sum(F.col(c).isNull().cast("int")) / F.lit(total) * 100).alias(c)
    for c in features_df.columns
]

null_percent = features_df.select(null_percent_exprs)
null_percent.show(truncate=False)

+-----------+----+-------+----------------+---------------+---------------------+----------------+------------+-----------+---------------+------------------------+---------------------+------------------+---------------+-----------------+-------------+
|customer_id|city|country|first_order_date|last_order_date|days_since_last_order|days_as_customer|total_orders|total_spent|avg_order_value|distinct_books_purchased|total_items_purchased|preferred_category|churn_label_60d|feature_timestamp|snapshot_date|
+-----------+----+-------+----------------+---------------+---------------------+----------------+------------+-----------+---------------+------------------------+---------------------+------------------+---------------+-----------------+-------------+
|0.0        |0.0 |0.0    |0.0             |0.0            |0.0                  |0.0             |0.0         |0.0        |0.0            |0.0                     |0.0                  |0.0               |0.0            |0.0              

## 8. Validación de duplicados a nivel de cliente

La capa `features` debería contener una fila por cliente para un `snapshot_date` determinado.  
Por tanto, la existencia de duplicados puede indicar un problema...

In [8]:
if "customer_id" in features_df.columns:
    duplicates_df = (
        features_df
        .groupBy("customer_id")
        .count()
        .filter(F.col("count") > 1)
    )

    print("Clientes duplicados:")
    duplicates_df.show(20, truncate=False)
else:
    print("No existe la columna customer_id en el dataset.")

Clientes duplicados:
+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+


## 9. Análisis descriptivo de variables numéricas

En esta sección se revisan estadísticas básicas de las variables numéricas.  
El objetivo es identificar rangos inesperados, posibles valores extremos o inconsistencias.

In [9]:
numeric_cols = [
    c for c, dtype in features_df.dtypes
    if dtype in ("int", "bigint", "double", "float", "decimal", "long")
]

print("Variables numéricas detectadas:")
for c in numeric_cols:
    print("-", c)

features_df.select(numeric_cols).describe().show(truncate=False)

Variables numéricas detectadas:
- days_since_last_order
- days_as_customer
- total_orders
- distinct_books_purchased
- total_items_purchased
- churn_label_60d
+-------+---------------------+----------------+-----------------+------------------------+---------------------+------------------+
|summary|days_since_last_order|days_as_customer|total_orders     |distinct_books_purchased|total_items_purchased|churn_label_60d   |
+-------+---------------------+----------------+-----------------+------------------------+---------------------+------------------+
|count  |403                  |403             |403              |403                     |403                  |403               |
|mean   |111.00992555831266   |2.0             |6.79652605459057 |12.846153846153847      |20.548387096774192   |0.5459057071960298|
|stddev |118.87312899373272   |0.0             |4.459133155524466|8.348810732250378       |13.927120760074706   |0.4985070856766576|
|min    |2                    |2           

In [11]:
for c in categorical_cols:
    print(f"\nDistribución de valores para: {c}")
    (
        features_df
        .groupBy(c)
        .count()
        .orderBy(F.desc("count"))
        .show(20, truncate=False)
    )


Distribución de valores para: customer_id
+------------------------------------+-----+
|customer_id                         |count|
+------------------------------------+-----+
|22430900-40eb-4912-968e-8f61f1ecd9b9|1    |
|dedff7d0-bb7b-4fc6-94a9-fbd9058a163c|1    |
|506beeff-64a7-4939-b570-9d9957d7d6f4|1    |
|13346bc1-3be9-4a6d-84c9-4bd8e5a0409b|1    |
|ba82dba4-84c5-4509-9c90-04c6158968d1|1    |
|2405519c-b9d6-415e-8fe0-847301fbf63a|1    |
|ffb8cd67-e125-4156-9f72-a2d12a04ff87|1    |
|d8a2155d-402f-4afd-b416-72d79b60f678|1    |
|fae2f737-3bab-4fdc-91b8-531f92848c34|1    |
|b3d785bc-1e75-4e3d-a324-cfcc8f03c231|1    |
|35ba59d7-c275-4372-9932-e0575aaf4b5f|1    |
|d066d441-485a-41ba-b471-3f9e657db37d|1    |
|8ff870a7-7aa1-4294-8f88-9389038f0b31|1    |
|149b2186-05fc-4c0f-a457-23141b9163a5|1    |
|01d9e8e6-975b-4777-8dba-87059485e5a3|1    |
|641f048a-ebd2-43d7-b197-d19730eef340|1    |
|1b5a2926-5cc8-4f73-b3f0-a782c3056d9d|1    |
|1c054e9a-3d85-438b-a6f3-ba73c7420f7f|1    |
|7978c785-7b

In [12]:
candidate_cols = [
    "days_since_last_order",
    "days_as_customer",
    "total_orders",
    "total_spent",
    "avg_order_value",
    "distinct_books_purchased",
    "total_items_purchased"
]

available_candidate_cols = [c for c in candidate_cols if c in features_df.columns]

if LABEL_COL in features_df.columns and available_candidate_cols:
    features_df.select([LABEL_COL] + available_candidate_cols).show(20, truncate=False)

    print("Promedios por clase:")
    features_df.groupBy(LABEL_COL).agg(
        *[F.avg(c).alias(f"avg_{c}") for c in available_candidate_cols]
    ).show(truncate=False)
else:
    print("No se encontraron las columnas necesarias para este análisis.")

+---------------+---------------------+----------------+------------+-----------+---------------+------------------------+---------------------+
|churn_label_60d|days_since_last_order|days_as_customer|total_orders|total_spent|avg_order_value|distinct_books_purchased|total_items_purchased|
+---------------+---------------------+----------------+------------+-----------+---------------+------------------------+---------------------+
|1              |121                  |2               |9           |1944.60    |216.066667     |22                      |35                   |
|1              |230                  |2               |4           |567.20     |141.800000     |9                       |9                    |
|0              |14                   |2               |2           |344.80     |172.400000     |5                       |6                    |
|1              |420                  |2               |2           |587.10     |293.550000     |5                       |8       

## Reflexión y validación del dataset

A lo largo de este notebook se ha realizado un proceso de exploración y validación de los datos generados en la capa `features`, a partir del pipeline de transformación definido en las secciones anteriores. En este contexto, analice de manera integral:

- La calidad de los datos generados  
- La consistencia entre variables  
- La presencia de valores nulos o inconsistencias  
- La coherencia entre las variables derivadas y la lógica de negocio  
- El tamaño del dataset y su representatividad 
- Entre otros aspectos que usted considere...

---

A partir de este análisis, determine si el dataset `customer_features` es adecuado para iniciar una fase de entrenamiento de modelos de Machine Learning.

En caso de identificar limitaciones, proponga qué ajustes serían necesarios a nivel de:

- pipeline de datos  
- construcción de features  
- definición de la variable objetivo  
- calidad y volumen de datos  

---

El objetivo de esta sección no es validar que el notebook funcione correctamente, sino:

> Evaluar si el proceso completo de generación de datos —desde la ingesta hasta la construcción de features— es confiable, reproducible y adecuado para soportar un sistema de Machine Learning en un entorno real.

---

**Nota:** La respuesta debe presentarse como un análisis técnico estructurado, no como una lista de respuestas.